In [2]:
# --- General ---
import time
import numpy as np
# --- SLM ---
import HEDS
from hedslib.heds_types import *

# --- Power control ---
from pylablib.devices.Thorlabs import MFF
from pylablib.devices.Thorlabs import PM160
import nidaqmx # ND filter motor
"""
# --- Rotation mount ---
from thorlabs_elliptec import ELLx, ELLError, ELLStatus, list_devices
import thorlabs_elliptec
"""

'\n# --- Rotation mount ---\nfrom thorlabs_elliptec import ELLx, ELLError, ELLStatus, list_devices\nimport thorlabs_elliptec\n'

In [2]:
# --- Instrument ---
from instruments.nd_filter_motor import NDFilterMotor
from instruments.slm import SLMController
from instruments.power_meter import PowerMeter
from instruments.flip_mount import FlipMount
#from instruments.rotation_mount import RotationMount
#from instruments.spectrometer import AndorSpectrometer

# --- Controller ---
from controllers.power_control import set_source_power, power_coversion, fix_power_conversion
#from controllers.polarization_scan import polarization_scan
#from controllers.polarized_spectroscopy import measure_polarized_spectra

In [3]:
import os
import numpy.random as rd
import matplotlib.pyplot as plt
from functools import partial

import pyvisa as pvisa
import qcodes as qc
from qcodes.instrument_drivers.yehlab_driver.QD import QD
import qcodes.instrument_drivers.nplab_drivers as npd
import qcodes.instrument_drivers.yehlab_driver as yd
from qcodes.instrument_drivers.stanford_research.SR830 import SR830
from qcodes.instrument_drivers.Keithley.Keithley_2636B import *

# Check address

In [4]:
rm = pvisa.ResourceManager()
resources = rm.list_resources()
print("List")
for res in resources:
    print(res)

List
GPIB0::13::INSTR
GPIB0::2::INSTR
GPIB0::3::INSTR
GPIB0::8::INSTR
GPIB1::3::INSTR
GPIB1::8::INSTR
PXI0::4-0.0::INSTR
USB0::0x05E6::0x2636::4353546::0::INSTR
USB0::0x05E6::0x2636::4437673::0::INSTR
USB0::0x0957::0x3D18::MY59030016::0::INSTR
USB0::0x1313::0x8078::P0005244::0::INSTR
USB0::0x1313::0x8078::PM001388::0::INSTR
USB0::0x2EC7::0x6522::802557043747210033::0::INSTR
ASRL3::INSTR
ASRL5::INSTR


In [5]:
nd = NDFilterMotor()
#slm = SLMController()
pm1 = PowerMeter('USB0::0x1313::0x8078::P0005244::0::INSTR')
pm2 = PowerMeter('USB0::0x1313::0x8078::PM001388::0::INSTR')
fm = FlipMount("37000705")
"""
rm1 = RotationMount()
rm1.home()
rm2 = RotationMount(port=rm1.stage, device_id=2)
rm2.home()
"""
#spec = AndorSpectrometer()

C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\pylablib\devices\Thorlabs\kinesis.py:230: UserWarning: model number MFF002 doesn't match the device ID prefix 37(MFF10.)
  warnings.warn("model number {} doesn't match the device ID prefix {}({})".format(model_no,port,port_model_no))


'\nrm1 = RotationMount()\nrm1.home()\nrm2 = RotationMount(port=rm1.stage, device_id=2)\nrm2.home()\n'

In [ ]:
# --- Parameters of spectrum ---
wavelength = 633

# --- Parameters of power meter
pm1.set_wavelength(wavelength)
print("Power meter: ", pm1.get_status())
pm2.set_wavelength(wavelength)
print("Power meter: ", pm2.get_status())

In [3]:
set_source_power(setpoint=10e-6,tolerance=10,flip_mount=fm,pm=pm1,motor=nd,tolerance_in_percent=True)

NameError: name 'fm' is not defined

In [5]:
sample_power=np.arange(10e-6,101e-6,10e-6)
print (sample_power)

[1.e-05 2.e-05 3.e-05 4.e-05 5.e-05 6.e-05 7.e-05 8.e-05 9.e-05 1.e-04]


In [ ]:
sample_power,source_power = power_conversion(sample_power, 10, fm,pm1, pm2, nd, tolerance_in_percent=True)
print("Sample power: ", sample_power)
print("Source power: ", source_power)

In [ ]:
# Close before restart the kernel
nd.close()
slm.close()
pm1.close()
pm2.close()
fm.close()
"""
rm1.close()
rm2.close()
"""
#spec.close()


In [22]:
# Set OAM and power test
slm.set_OAM(5, shift_x = 300, shift_y = 0)
set_power(setpoint=90e-6,tolerance=1e-6,flip_mount=fm,pm=pm1,motor=nd)

# 2636B

In [5]:
k2636B = Keithley2600('k2636B','USB0::0x05E6::0x2636::4353546::0::INSTR')

Connected to: Keithley Instruments Inc. 2636B (serial:4353546, firmware:3.3.5) in 0.16s


In [6]:
smua_mode = k2636B.smua.mode()
print(f"SMUA Mode: {smua_mode}")
smub_mode = k2636B.smub.mode()
print(f"SMUB Mode: {smub_mode}")

SMUA Mode: voltage
SMUB Mode: voltage


In [27]:
# 設置電壓源和測量
k2636B.smua.limiti(10e-6)  # 設置電流上限
k2636B.smua.measure_autorange_i_enabled(1)  # 啟用自動範圍
k2636B.smua.sourcerange_v(20)  # 設置電壓源範圍

# 設置閘極電壓源和測量
k2636B.smub.limiti(10e-6)  # 設置電流上限
k2636B.smub.measure_autorange_i_enabled(1)  # 啟用自動範圍
k2636B.smub.sourcerange_v(20)  # 設置電壓源範圍


In [23]:
def sweep_range(start, stop, step, dual_sweep=False, SR830=False):
    vol_val = np.arange(start, stop + step, step)

    if start * step < 0:
        point0 = int(start / step)
        vol_val[point0 - 1]=0
        
    if SR830 == True:
        for i in range (len(vol_val)):
            if vol_val[i] <= 0.004:
                vol_val[i] = 0.004
    
    if dual_sweep == True:
        vol_val = np.append(np.flip(vol_val), vol_val)
    print(vol_val)
    return vol_val

In [24]:
print("Vd: ")
vd_range = sweep_range(-3,3,0.2,dual_sweep=True)
print("Vg: ")
vg_range = sweep_range(-10,10,2)
print("V_AC: ")
vac_range = sweep_range(0,3,0.05,dual_sweep=True,SR830=True)

Vd: 
[ 3.   2.8  2.6  2.4  2.2  2.   1.8  1.6  1.4  1.2  1.   0.8  0.6  0.4
  0.2  0.  -0.2 -0.4 -0.6 -0.8 -1.  -1.2 -1.4 -1.6 -1.8 -2.  -2.2 -2.4
 -2.6 -2.8 -3.  -3.  -2.8 -2.6 -2.4 -2.2 -2.  -1.8 -1.6 -1.4 -1.2 -1.
 -0.8 -0.6 -0.4 -0.2  0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8
  2.   2.2  2.4  2.6  2.8  3. ]
Vg: 
[-10  -8  -6  -4  -2   0   2   4   6   8  10]
V_AC: 
[3.    2.95  2.9   2.85  2.8   2.75  2.7   2.65  2.6   2.55  2.5   2.45
 2.4   2.35  2.3   2.25  2.2   2.15  2.1   2.05  2.    1.95  1.9   1.85
 1.8   1.75  1.7   1.65  1.6   1.55  1.5   1.45  1.4   1.35  1.3   1.25
 1.2   1.15  1.1   1.05  1.    0.95  0.9   0.85  0.8   0.75  0.7   0.65
 0.6   0.55  0.5   0.45  0.4   0.35  0.3   0.25  0.2   0.15  0.1   0.05
 0.004 0.004 0.05  0.1   0.15  0.2   0.25  0.3   0.35  0.4   0.45  0.5
 0.55  0.6   0.65  0.7   0.75  0.8   0.85  0.9   0.95  1.    1.05  1.1
 1.15  1.2   1.25  1.3   1.35  1.4   1.45  1.5   1.55  1.6   1.65  1.7
 1.75  1.8   1.85  1.9   1.95  2.    2.05  2.1   

# SR830

In [37]:
sr830=SR830('lockin1','GPIB1::8::INSTR')

Connected to: Stanford_Research_Systems SR830 (serial:s/n92421, firmware:ver1.07) in 0.15s


C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument lockin1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument lockin1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


In [38]:
sr830.write("FMOD 1")     # 内部参考模式
sr830.write("FREQ 311")   # 参考频率 (Hz)
sr830.write("SLVL 0.04")   # 输出电压振幅 (V)
sr830.write("ISRC 0")     # 模式（0 = Voltage, 1 = Current）
"""
volt_val = np.linspace(0.04, 1, 100)
k2636B.smub.volt(20)
k2636B.smub.output(1)
sr830.write("SLVL 1") 

sr830.write('AGAN')		  #自動檔位 sensitivity
"""

'\nvolt_val = np.linspace(0.04, 1, 100)\nk2636B.smub.volt(20)\nk2636B.smub.output(1)\nsr830.write("SLVL 1") \n\nsr830.write(\'AGAN\')\t\t  #自動檔位 sensitivity\n'

# Measurement loop

In [ ]:
OAM = [-3, -2, -1, 0, 1, 2, 3]
Power = [1.72e-3, 1.74e-3, 1.75e-3, 1.75e-3, 1.76e-3, 1.75e-3, 1.73e-3] #Power for each OAM
tolerance = 1e-6 
#cycle_number = 40


for i in range(len(OAM)):
    set_OAM(OAM[i], shift_x = 150, shift_y = 0)
    setpoint = Power[i]
    set_power(setpoint, 1e-6)
    #lOCK MEASUREMENT
    
    time.sleep(1)
    flip_mount.move_to_state(0) #Block
    
   

Measured: 0.001798 W, Error: -4.839495e-05 W
Measured: 0.001716 W, Error: 3.408170e-05 W
Measured: 0.001753 W, Error: -3.133610e-06 W
Measured: 0.001749 W, Error: 8.895900e-07 W
Target reached.
Measured: 0.001758 W, Error: -8.498140e-06 W
Measured: 0.000038 W, Error: 1.712281e-03 W
Measured: 0.000028 W, Error: 1.722495e-03 W
Measured: 0.000045 W, Error: 1.705284e-03 W
Measured: 0.000027 W, Error: 1.722855e-03 W
Measured: 0.000039 W, Error: 1.711335e-03 W
Measured: 0.000034 W, Error: 1.716460e-03 W
Measured: 0.000030 W, Error: 1.719681e-03 W
Measured: 0.000020 W, Error: 1.730075e-03 W
Measured: 0.000046 W, Error: 1.704195e-03 W
Measured: 0.000021 W, Error: 1.728673e-03 W
Measured: 0.000020 W, Error: 1.730208e-03 W
Measured: 0.000037 W, Error: 1.712531e-03 W
Measured: 0.000028 W, Error: 1.721972e-03 W
Measured: 0.000024 W, Error: 1.726481e-03 W
Measured: 0.000030 W, Error: 1.719521e-03 W
Measured: 0.000022 W, Error: 1.728360e-03 W
Measured: 0.000046 W, Error: 1.703902e-03 W
Measured: 0.0

Measured: 0.001739 W, Error: -9.387400e-06 W
Measured: 0.001778 W, Error: -4.760830e-05 W
Measured: 0.001765 W, Error: -3.453264e-05 W
Measured: 0.001681 W, Error: 4.861467e-05 W
Measured: 0.001719 W, Error: 1.072870e-05 W
Measured: 0.001669 W, Error: 6.068426e-05 W
Measured: 0.001746 W, Error: -1.575776e-05 W
Measured: 0.001677 W, Error: 5.263787e-05 W
Measured: 0.001805 W, Error: -7.543598e-05 W
Measured: 0.001766 W, Error: -3.587387e-05 W
Measured: 0.001666 W, Error: 6.403691e-05 W
Measured: 0.001763 W, Error: -3.252134e-05 W
Measured: 0.001728 W, Error: 1.676250e-06 W
Measured: 0.001694 W, Error: 3.620969e-05 W
Measured: 0.001746 W, Error: -1.642797e-05 W
Measured: 0.001782 W, Error: -5.196677e-05 W
Measured: 0.001726 W, Error: 3.688140e-06 W
Measured: 0.001699 W, Error: 3.118055e-05 W
Measured: 0.001767 W, Error: -3.687981e-05 W
Measured: 0.001768 W, Error: -3.822057e-05 W
Measured: 0.001711 W, Error: 1.944576e-05 W
Measured: 0.001699 W, Error: 3.084527e-05 W
Measured: 0.001752 W,

# Parameters

In [39]:
vd_val = np.linspace(0.004, 1, 100)
vd_val = np.append(vd_val, np.flip(vd_val))
vg_val = np.arange(10, -12, -2)
print(vd_val)
print(vg_val)

[0.004      0.01406061 0.02412121 0.03418182 0.04424242 0.05430303
 0.06436364 0.07442424 0.08448485 0.09454545 0.10460606 0.11466667
 0.12472727 0.13478788 0.14484848 0.15490909 0.1649697  0.1750303
 0.18509091 0.19515152 0.20521212 0.21527273 0.22533333 0.23539394
 0.24545455 0.25551515 0.26557576 0.27563636 0.28569697 0.29575758
 0.30581818 0.31587879 0.32593939 0.336      0.34606061 0.35612121
 0.36618182 0.37624242 0.38630303 0.39636364 0.40642424 0.41648485
 0.42654545 0.43660606 0.44666667 0.45672727 0.46678788 0.47684848
 0.48690909 0.4969697  0.5070303  0.51709091 0.52715152 0.53721212
 0.54727273 0.55733333 0.56739394 0.57745455 0.58751515 0.59757576
 0.60763636 0.61769697 0.62775758 0.63781818 0.64787879 0.65793939
 0.668      0.67806061 0.68812121 0.69818182 0.70824242 0.71830303
 0.72836364 0.73842424 0.74848485 0.75854545 0.76860606 0.77866667
 0.78872727 0.79878788 0.80884848 0.81890909 0.8289697  0.8390303
 0.84909091 0.85915152 0.86921212 0.87927273 0.88933333 0.899393

# Auto Range and Phase

In [40]:
set_OAM(0, shift_x = 300, shift_y = 0)
set_power(1.75e-3, 1e-6)

k2636B.smub.volt(10)
time.sleep(1)
k2636B.smub.output(1)
sr830.write("SLVL 1") 

time.sleep(1)
sr830.write('AGAN')		  #自動檔位 sensitivity
time.sleep(1)
sr830.write('APHS')
time.sleep(1)

k2636B.smub.volt(0)
time.sleep(1)
sr830.write("SLVL 0.004")
time.sleep(1)

Measured: 0.000030 W, Error: 1.719904e-03 W
Measured: 0.000040 W, Error: 1.709886e-03 W
Measured: 0.000053 W, Error: 1.696562e-03 W
Measured: 0.000026 W, Error: 1.723804e-03 W
Measured: 0.000042 W, Error: 1.707838e-03 W
Measured: 0.000050 W, Error: 1.699762e-03 W
Measured: 0.000025 W, Error: 1.725276e-03 W
Measured: 0.000043 W, Error: 1.707255e-03 W
Measured: 0.000044 W, Error: 1.706080e-03 W
Measured: 0.000042 W, Error: 1.708018e-03 W
Measured: 0.000031 W, Error: 1.719481e-03 W
Measured: 0.000035 W, Error: 1.714855e-03 W
Measured: 0.000022 W, Error: 1.728340e-03 W
Measured: 0.000042 W, Error: 1.708408e-03 W
Measured: 0.000019 W, Error: 1.730514e-03 W
Measured: 0.000041 W, Error: 1.708538e-03 W
Measured: 0.000021 W, Error: 1.728689e-03 W
Measured: 0.000038 W, Error: 1.711944e-03 W
Measured: 0.000040 W, Error: 1.710149e-03 W
Measured: 0.000036 W, Error: 1.714089e-03 W
Measured: 0.000037 W, Error: 1.713383e-03 W
Measured: 0.000112 W, Error: 1.638215e-03 W
Measured: 0.000132 W, Error: 1.6

Measured: 0.000277 W, Error: 1.473335e-03 W
Measured: 0.000277 W, Error: 1.473035e-03 W
Measured: 0.000269 W, Error: 1.480770e-03 W
Measured: 0.000285 W, Error: 1.464800e-03 W
Measured: 0.000280 W, Error: 1.470068e-03 W
Measured: 0.000288 W, Error: 1.461633e-03 W
Measured: 0.000273 W, Error: 1.476836e-03 W
Measured: 0.000298 W, Error: 1.452364e-03 W
Measured: 0.000280 W, Error: 1.469501e-03 W
Measured: 0.000284 W, Error: 1.465500e-03 W
Measured: 0.000297 W, Error: 1.453031e-03 W
Measured: 0.000272 W, Error: 1.478103e-03 W
Measured: 0.000283 W, Error: 1.466634e-03 W
Measured: 0.000304 W, Error: 1.446196e-03 W
Measured: 0.000282 W, Error: 1.468034e-03 W
Measured: 0.000273 W, Error: 1.477203e-03 W
Measured: 0.000290 W, Error: 1.459899e-03 W
Measured: 0.000300 W, Error: 1.450197e-03 W
Measured: 0.000295 W, Error: 1.455498e-03 W
Measured: 0.000302 W, Error: 1.447763e-03 W
Measured: 0.000310 W, Error: 1.439661e-03 W
Measured: 0.000306 W, Error: 1.444462e-03 W
Measured: 0.000297 W, Error: 1.4

Measured: 0.000603 W, Error: 1.146593e-03 W
Measured: 0.000611 W, Error: 1.138724e-03 W
Measured: 0.000629 W, Error: 1.121187e-03 W
Measured: 0.000611 W, Error: 1.138524e-03 W
Measured: 0.000643 W, Error: 1.107417e-03 W
Measured: 0.000616 W, Error: 1.134023e-03 W
Measured: 0.000645 W, Error: 1.105116e-03 W
Measured: 0.000620 W, Error: 1.130089e-03 W
Measured: 0.000623 W, Error: 1.126621e-03 W
Measured: 0.000651 W, Error: 1.099282e-03 W
Measured: 0.000625 W, Error: 1.124787e-03 W
Measured: 0.000622 W, Error: 1.128355e-03 W
Measured: 0.000654 W, Error: 1.096047e-03 W
Measured: 0.000623 W, Error: 1.126755e-03 W
Measured: 0.000626 W, Error: 1.124087e-03 W
Measured: 0.000641 W, Error: 1.108817e-03 W
Measured: 0.000624 W, Error: 1.125588e-03 W
Measured: 0.000647 W, Error: 1.102716e-03 W
Measured: 0.000631 W, Error: 1.119286e-03 W
Measured: 0.000646 W, Error: 1.103549e-03 W
Measured: 0.000629 W, Error: 1.120820e-03 W
Measured: 0.000622 W, Error: 1.127922e-03 W
Measured: 0.000650 W, Error: 1.0

Measured: 0.001080 W, Error: 6.700148e-04 W
Measured: 0.001078 W, Error: 6.715819e-04 W
Measured: 0.001094 W, Error: 6.564116e-04 W
Measured: 0.001079 W, Error: 6.713485e-04 W
Measured: 0.001084 W, Error: 6.656805e-04 W
Measured: 0.001094 W, Error: 6.559782e-04 W
Measured: 0.001084 W, Error: 6.658472e-04 W
Measured: 0.001091 W, Error: 6.590122e-04 W
Measured: 0.001098 W, Error: 6.521439e-04 W
Measured: 0.001103 W, Error: 6.473428e-04 W
Measured: 0.001101 W, Error: 6.491765e-04 W
Measured: 0.001089 W, Error: 6.607127e-04 W
Measured: 0.001089 W, Error: 6.614461e-04 W
Measured: 0.001098 W, Error: 6.522107e-04 W
Measured: 0.001104 W, Error: 6.458759e-04 W
Measured: 0.001094 W, Error: 6.558782e-04 W
Measured: 0.001107 W, Error: 6.428419e-04 W
Measured: 0.001099 W, Error: 6.513771e-04 W
Measured: 0.001111 W, Error: 6.391410e-04 W
Measured: 0.001117 W, Error: 6.325060e-04 W
Measured: 0.001112 W, Error: 6.376405e-04 W
Measured: 0.001124 W, Error: 6.264713e-04 W
Measured: 0.001128 W, Error: 6.2

In [42]:
set_OAM(0)

In [65]:
OAM = [-3, -2, -1, 0, 1, 2, 3]
Power = [1.72e-3, 1.74e-3, 1.75e-3, 1.75e-3, 1.76e-3, 1.75e-3, 1.73e-3] #Power for each OAM
tolerance = 1e-6 


for vg in vg_val:
    k2636B.smub.volt(vg)
    k2636B.smub.output(1)
    sr830.write("SLVL 0.004")
    time.sleep(1)

    for i in range(len(OAM)):
        set_OAM(OAM[i], shift_x = 300, shift_y = 0)
        setpoint = Power[i]
        set_power(setpoint, 1e-6)
        k2636B.smub.output(1)
        
        #lOCK MEASUREMENT
        time.sleep(5)

        data1 , plot1=npd.single_param_sweep(sr830.amplitude, vd_val, 0.2, sr830.R, sr830.P, sr830.X, DataName=f'AC_VG = {vg}_noQWP_OAM = {OAM[i]}')
        time.sleep(1)
        sr830.write("SLVL 0.004")
        #k2636B.smub.volt(0)
        k2636B.smub.output(0)
        flip_mount.move_to_state(0) #Block
        time.sleep(60)


Measured: 0.001714 W, Error: 5.757840e-06 W
Measured: 0.001724 W, Error: -4.299970e-06 W
Measured: 0.001630 W, Error: 8.957571e-05 W
Measured: 0.001629 W, Error: 9.125232e-05 W
Measured: 0.001633 W, Error: 8.689373e-05 W
Measured: 0.001708 W, Error: 1.179293e-05 W
Measured: 0.001657 W, Error: 6.308925e-05 W
Measured: 0.001700 W, Error: 1.983932e-05 W
Measured: 0.001705 W, Error: 1.514546e-05 W
Measured: 0.001728 W, Error: -7.988360e-06 W
Measured: 0.001727 W, Error: -7.317810e-06 W
Measured: 0.001723 W, Error: -2.959220e-06 W
Measured: 0.001689 W, Error: 3.056828e-05 W
Measured: 0.001655 W, Error: 6.476586e-05 W
Measured: 0.001695 W, Error: 2.486847e-05 W
Measured: 0.001688 W, Error: 3.190892e-05 W
Measured: 0.001694 W, Error: 2.587441e-05 W
Measured: 0.001695 W, Error: 2.520374e-05 W
Measured: 0.001713 W, Error: 6.763790e-06 W
Measured: 0.001745 W, Error: -2.475193e-05 W
Measured: 0.001763 W, Error: -4.285661e-05 W
Measured: 0.001762 W, Error: -4.185067e-05 W
Measured: 0.001746 W, Err

Measured: 0.001777 W, Error: -2.660235e-05 W
Measured: 0.001772 W, Error: -2.190895e-05 W
Measured: 0.001717 W, Error: 3.341114e-05 W
Measured: 0.001730 W, Error: 1.966494e-05 W
Measured: 0.001765 W, Error: -1.453264e-05 W
Measured: 0.001740 W, Error: 9.606650e-06 W
Measured: 0.001791 W, Error: -4.135450e-05 W
Measured: 0.001781 W, Error: -3.096083e-05 W
Measured: 0.001774 W, Error: -2.425565e-05 W
Measured: 0.001758 W, Error: -8.162750e-06 W
Measured: 0.001754 W, Error: -4.139550e-06 W
Measured: 0.001749 W, Error: 1.224870e-06 W
Measured: 0.001701 W, Error: 4.916866e-05 W
Measured: 0.001736 W, Error: 1.362985e-05 W
Measured: 0.001732 W, Error: 1.765305e-05 W
Measured: 0.001765 W, Error: -1.453264e-05 W
Measured: 0.001724 W, Error: 2.603484e-05 W
Measured: 0.001724 W, Error: 2.637011e-05 W
Measured: 0.001789 W, Error: -3.900780e-05 W
Measured: 0.001731 W, Error: 1.899427e-05 W
Measured: 0.001747 W, Error: 3.236760e-06 W
Measured: 0.001717 W, Error: 3.341114e-05 W
Measured: 0.001825 W, 

Measured: 0.001775 W, Error: -3.492632e-05 W
Measured: 0.001758 W, Error: -1.816275e-05 W
Measured: 0.001710 W, Error: 2.978104e-05 W
Measured: 0.001723 W, Error: 1.704078e-05 W
Measured: 0.001754 W, Error: -1.380427e-05 W
Measured: 0.001736 W, Error: 3.629850e-06 W
Measured: 0.001782 W, Error: -4.196677e-05 W
Measured: 0.001708 W, Error: 3.246348e-05 W
Measured: 0.001744 W, Error: -4.416540e-06 W
Measured: 0.001735 W, Error: 4.635800e-06 W
Measured: 0.001719 W, Error: 2.106398e-05 W
Measured: 0.001800 W, Error: -6.040684e-05 W
Measured: 0.001775 W, Error: -3.459093e-05 W
Measured: 0.001770 W, Error: -2.956179e-05 W
Measured: 0.001736 W, Error: 3.629850e-06 W
Measured: 0.001726 W, Error: 1.368814e-05 W
Measured: 0.001716 W, Error: 2.441709e-05 W
Measured: 0.001749 W, Error: -9.445680e-06 W
Measured: 0.001756 W, Error: -1.615097e-05 W
Measured: 0.001767 W, Error: -2.687981e-05 W
Measured: 0.001744 W, Error: -3.745990e-06 W
Measured: 0.001667 W, Error: 7.303096e-05 W
Measured: 0.001724 W

Measured: 0.001735 W, Error: 1.463580e-05 W
Measured: 0.001781 W, Error: -3.096083e-05 W
Measured: 0.001738 W, Error: 1.228910e-05 W
Measured: 0.001809 W, Error: -5.945918e-05 W
Measured: 0.001828 W, Error: -7.756386e-05 W
Measured: 0.001835 W, Error: -8.527498e-05 W
Measured: 0.001753 W, Error: -3.469000e-06 W
Measured: 0.001848 W, Error: -9.801535e-05 W
Measured: 0.001833 W, Error: -8.292839e-05 W
Measured: 0.001790 W, Error: -4.034856e-05 W
Measured: 0.001809 W, Error: -5.912390e-05 W
Measured: 0.001805 W, Error: -5.510071e-05 W
Measured: 0.001787 W, Error: -3.699591e-05 W
Measured: 0.001738 W, Error: 1.228910e-05 W
Measured: 0.001746 W, Error: 3.907310e-06 W
Measured: 0.001765 W, Error: -1.486804e-05 W
Measured: 0.001742 W, Error: 8.265900e-06 W
Measured: 0.001789 W, Error: -3.900780e-05 W
Measured: 0.001782 W, Error: -3.230205e-05 W
Measured: 0.001763 W, Error: -1.319189e-05 W
Measured: 0.001722 W, Error: 2.771133e-05 W
Measured: 0.001764 W, Error: -1.386209e-05 W
Measured: 0.0017

Measured: 0.001743 W, Error: 6.924680e-06 W
Measured: 0.001752 W, Error: -2.463050e-06 W
Measured: 0.001734 W, Error: 1.597702e-05 W
Measured: 0.001764 W, Error: -1.419737e-05 W
Measured: 0.001781 W, Error: -3.129622e-05 W
Measured: 0.001712 W, Error: 3.844029e-05 W
Measured: 0.001770 W, Error: -2.023245e-05 W
Measured: 0.001751 W, Error: -1.457110e-06 W
Measured: 0.001792 W, Error: -4.168978e-05 W
Measured: 0.001749 W, Error: 8.895900e-07 W
Target reached.
Measured: 0.001746 W, Error: 3.907310e-06 W
Started at 2025-09-23 00:10:25
DataSet:
   location = 'data/2025-09-23/#005_AC_VG = 8_noQWP_OAM = 2_00-10-22'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (200,)
   Measured | lockin1_R             | R            | (200,)
   Measured | lockin1_P             | P            | (200,)
   Measured | lockin1_X             | X            | (200,)
Finished at 2025-09-23 00:11:38
Measured: 0.001763 W, Error: -3.252134e-05 W


...

Measured: 0.001708 W, Error: 4.246348e-05 W
Measured: 0.001788 W, Error: -3.833713e-05 W
Measured: 0.001779 W, Error: -2.894952e-05 W
Measured: 0.001728 W, Error: 2.201164e-05 W
Measured: 0.001726 W, Error: 2.435880e-05 W
Measured: 0.001792 W, Error: -4.236033e-05 W
Measured: 0.001705 W, Error: 4.514546e-05 W
Measured: 0.001736 W, Error: 1.396513e-05 W
Measured: 0.001765 W, Error: -1.520331e-05 W
Measured: 0.001740 W, Error: 9.606650e-06 W
Measured: 0.001801 W, Error: -5.107739e-05 W
Measured: 0.001780 W, Error: -2.995546e-05 W
Measured: 0.001746 W, Error: 3.572030e-06 W
Measured: 0.001738 W, Error: 1.195382e-05 W
Measured: 0.001809 W, Error: -5.878851e-05 W
Measured: 0.001708 W, Error: 4.246348e-05 W
Measured: 0.001839 W, Error: -8.862762e-05 W
Measured: 0.001741 W, Error: 8.600710e-06 W
Measured: 0.001755 W, Error: -5.145030e-06 W
Measured: 0.001784 W, Error: -3.364327e-05 W
Measured: 0.001712 W, Error: 3.810501e-05 W
Measured: 0.001807 W, Error: -5.677720e-05 W
Measured: 0.001719 W,

Measured: 0.001762 W, Error: -3.218594e-05 W
Measured: 0.001778 W, Error: -4.794357e-05 W
Measured: 0.001723 W, Error: 7.040780e-06 W
Measured: 0.001668 W, Error: 6.169021e-05 W
Measured: 0.001790 W, Error: -6.034856e-05 W
Measured: 0.001710 W, Error: 2.045159e-05 W
Measured: 0.001763 W, Error: -3.252134e-05 W
Measured: 0.001681 W, Error: 4.861467e-05 W
Measured: 0.001743 W, Error: -1.307532e-05 W
Measured: 0.001698 W, Error: 3.185121e-05 W
Measured: 0.001754 W, Error: -2.447483e-05 W
Measured: 0.001747 W, Error: -1.743391e-05 W
Measured: 0.001707 W, Error: 2.346896e-05 W
Measured: 0.001695 W, Error: 3.486847e-05 W
Measured: 0.001730 W, Error: 2.100000e-10 W
Target reached.
Measured: 0.001741 W, Error: -1.072862e-05 W
Started at 2025-09-23 02:14:58
DataSet:
   location = 'data/2025-09-23/#055_AC_VG = -6_noQWP_OAM = 3_02-14-56'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (200,)
   Measured | lockin1_R           

Measured: 0.001739 W, Error: 2.061260e-05 W
Measured: 0.001757 W, Error: 3.178470e-06 W
Measured: 0.001809 W, Error: -4.945918e-05 W
Measured: 0.001797 W, Error: -3.705420e-05 W
Measured: 0.001771 W, Error: -1.056773e-05 W
Measured: 0.001780 W, Error: -1.995546e-05 W
Measured: 0.001783 W, Error: -2.330799e-05 W
Measured: 0.001748 W, Error: 1.223081e-05 W
Measured: 0.001726 W, Error: 3.435880e-05 W
Measured: 0.001770 W, Error: -9.561790e-06 W
Measured: 0.001787 W, Error: -2.699591e-05 W
Measured: 0.001755 W, Error: 4.519690e-06 W
Measured: 0.001720 W, Error: 3.972322e-05 W
Measured: 0.001753 W, Error: 6.866390e-06 W
Measured: 0.001745 W, Error: 1.491279e-05 W
Measured: 0.001746 W, Error: 1.390731e-05 W
Measured: 0.001780 W, Error: -1.962007e-05 W
Measured: 0.001778 W, Error: -1.794357e-05 W
Measured: 0.001775 W, Error: -1.459093e-05 W
Measured: 0.001711 W, Error: 4.877556e-05 W
Measured: 0.001740 W, Error: 1.994193e-05 W
Measured: 0.001773 W, Error: -1.324971e-05 W
Measured: 0.001749 W,

Measured: 0.001731 W, Error: -1.067034e-05 W
Measured: 0.001729 W, Error: -8.658560e-06 W
Measured: 0.001743 W, Error: -2.307532e-05 W
Measured: 0.001708 W, Error: 1.212821e-05 W
Measured: 0.001665 W, Error: 5.470758e-05 W
Measured: 0.001692 W, Error: 2.788572e-05 W
Measured: 0.001708 W, Error: 1.246348e-05 W
Measured: 0.001712 W, Error: 8.105010e-06 W
Measured: 0.001755 W, Error: -3.514503e-05 W
Measured: 0.001747 W, Error: -2.709852e-05 W
Measured: 0.001755 W, Error: -3.480964e-05 W
Measured: 0.001714 W, Error: 6.428400e-06 W
Measured: 0.001770 W, Error: -4.956179e-05 W
Measured: 0.001718 W, Error: 2.405200e-06 W
Measured: 0.001723 W, Error: -2.623940e-06 W
Measured: 0.001675 W, Error: 4.531984e-05 W
Measured: 0.001731 W, Error: -1.067034e-05 W
Measured: 0.001728 W, Error: -7.988360e-06 W
Measured: 0.001689 W, Error: 3.056828e-05 W
Measured: 0.001713 W, Error: 7.099060e-06 W
Measured: 0.001745 W, Error: -2.508721e-05 W
Measured: 0.001730 W, Error: -9.664510e-06 W
Measured: 0.001707 W

Measured: 0.001760 W, Error: -9.838780e-06 W
Measured: 0.001776 W, Error: -2.626754e-05 W
Measured: 0.001775 W, Error: -2.526160e-05 W
Measured: 0.001769 W, Error: -1.922651e-05 W
Measured: 0.001769 W, Error: -1.922651e-05 W
Measured: 0.001765 W, Error: -1.520331e-05 W
Measured: 0.001776 W, Error: -2.593215e-05 W
Measured: 0.001771 W, Error: -2.123840e-05 W
Measured: 0.001789 W, Error: -3.867253e-05 W
Measured: 0.001752 W, Error: -2.127660e-06 W
Measured: 0.001870 W, Error: -1.201432e-04 W
Measured: 0.001789 W, Error: -3.900780e-05 W
Measured: 0.001807 W, Error: -5.711201e-05 W
Measured: 0.001760 W, Error: -9.838780e-06 W
Measured: 0.001736 W, Error: 1.362985e-05 W
Measured: 0.001782 W, Error: -3.163149e-05 W
Measured: 0.001802 W, Error: -5.208334e-05 W
Measured: 0.001749 W, Error: 8.895900e-07 W
Target reached.
Measured: 0.001712 W, Error: 3.810501e-05 W
Started at 2025-09-23 02:40:48
DataSet:
   location = 'data/2025-09-23/#066_AC_VG = -10_noQWP_OAM = 0_02-40-46'
   <Type>   | <array

In [49]:
#Define the input values (Vd)
vol_val = np.arange(-1, 1.01, 0.05)
vol_val[20]=0
print(np.shape(vol_val))
print(vol_val)

SyntaxError: EOF while scanning triple-quoted string literal (2176674328.py, line 6)

In [52]:

vol_val = np.arange(-1, 1.01, 0.05)
vol_val[20] = 0
x = np.arange(1, -1.01, -0.05)
x[20] = 0
vol_val = np.append(vol_val, x)
print(vol_val)

"""
print(x)


y = -1*vol_val
vol_val = np.append(vol_val, y)
print(np.shape(vol_val))
print(vol_val)
re_vol_val = vol_val* -1
print(re_vol_val)
"""

[-1.   -0.95 -0.9  -0.85 -0.8  -0.75 -0.7  -0.65 -0.6  -0.55 -0.5  -0.45
 -0.4  -0.35 -0.3  -0.25 -0.2  -0.15 -0.1  -0.05  0.    0.05  0.1   0.15
  0.2   0.25  0.3   0.35  0.4   0.45  0.5   0.55  0.6   0.65  0.7   0.75
  0.8   0.85  0.9   0.95  1.    1.    0.95  0.9   0.85  0.8   0.75  0.7
  0.65  0.6   0.55  0.5   0.45  0.4   0.35  0.3   0.25  0.2   0.15  0.1
  0.05  0.   -0.05 -0.1  -0.15 -0.2  -0.25 -0.3  -0.35 -0.4  -0.45 -0.5
 -0.55 -0.6  -0.65 -0.7  -0.75 -0.8  -0.85 -0.9  -0.95 -1.  ]


'\nprint(x)\n\n\ny = -1*vol_val\nvol_val = np.append(vol_val, y)\nprint(np.shape(vol_val))\nprint(vol_val)\nre_vol_val = vol_val* -1\nprint(re_vol_val)\n'

In [65]:
k2636B.smua.volt(-1)
k2636B.smub.volt(0)
time.sleep(1)
k2636B.smua.output(1)
k2636B.smub.output(1)




In [61]:
sr830.write('AGAN')
time.sleep(5)
sr830.write('AGAN')

In [93]:
k2636B.smua.volt(-2)
k2636B.smub.volt(0)

In [18]:
OAM = [-3, -2, -1, 0, 1, 2, 3]
Power = [1.72e-3, 1.74e-3, 1.75e-3, 1.75e-3, 1.76e-3, 1.75e-3, 1.73e-3] #Power for each OAM
tolerance = 1e-6 
#cycle_number = 40


for i in range(len(OAM)):
    set_OAM(OAM[i], shift_x = 300, shift_y = 0)
    setpoint = Power[i]
    set_power(setpoint, 1e-6)
    #lOCK MEASUREMENT
    k2636B.smua.volt(-2)
    k2636B.smub.volt(20)
    time.sleep(1)
    k2636B.smua.output(1)
    k2636B.smub.output(1)
    #sr830.write('AGAN')
    #time.sleep(5)
    #sr830.write('AGAN')
    #sr830.increment_sensitivity()
    #k2636B.smua.output(0)
    #k2636B.smub.output(0)
    #k2636B.smua.volt(0)
    time.sleep(2)
    #k2636B.smua.output(1)
    #k2636B.smub.output(1)
    #time.sleep(1)
    data1 , plot1=npd.single_param_sweep(k2636B.smua.volt, vol_val, 0.2, k2636B.smua.curr, k2636B.smua.volt, k2636B.smub.volt, sr830.R, sr830.P, sr830.X, DataName=f'test_OAM = {OAM[i]}')
    time.sleep(1)
    k2636B.smua.volt(0)
    k2636B.smub.volt(0)
    k2636B.smua.output(0)
    k2636B.smub.output(0)
    
    time.sleep(1)
    flip_mount.move_to_state(0) #Block
    
   

Measured: 0.001721 W, Error: -6.120500e-07 W
Target reached.
Measured: 0.001754 W, Error: -3.413955e-05 W


NameError: name 'vol_val' is not defined

In [110]:
sr830.write("SLVL 0.04")   # 输出电压振幅 (V)

In [96]:
k2636B.smua.output(0)
#k2636B.smub.output(0)
k2636B.smua.volt(0)
time.sleep(2)
k2636B.smua.output(1)
k2636B.smub.output(1)
time.sleep(1)
data1 , plot1=npd.single_param_sweep(k2636B.smua.volt, vol_val, 1.5, k2636B.smua.curr, k2636B.smua.volt, k2636B.smub.volt, sr830.R, sr830.P, sr830.X, DataName='test')
time.sleep(1)
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-09-18 16:49:21
DataSet:
   location = 'data/2025-09-18/#016_test_16-49-18'
   <Type>   | <array_id>           | <array.name> | <array.shape>
   Setpoint | k2636B_smua_volt_set | volt         | (82,)
   Measured | k2636B_smua_curr     | curr         | (82,)
   Measured | k2636B_smua_volt     | volt         | (82,)
   Measured | k2636B_smub_volt     | volt         | (82,)
   Measured | lockin1_R            | R            | (82,)
   Measured | lockin1_P            | P            | (82,)
   Measured | lockin1_X            | X            | (82,)
Finished at 2025-09-18 16:51:46


In [102]:
sr830.increment_sensitivity()

KeyError: (16, 'getting lockin1_input_config', 'getting lockin1_sensitivity')

In [59]:
k2636B.smua.output(1)
k2636B.smub.output(1)
time.sleep(1)
data1, plot1=npd.single_param_sweep(k2636B.smua.volt, vol_val, 0.002, k2636B.smua.curr, k2636B.smua.volt, k2636B.smub.volt, k2636B.smub.curr, sr830.R, sr830.P, sr830.X, DataName='test') #,sr830.R, sr830.P, sr830.X, DataName='dark')
time.sleep(1)
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-11-04 19:32:35
Received SIGINT, Will interrupt at first suitable time. Send second SIGINT to interrupt immediately.
DataSet:
   location = 'data/2025-11-04/#005_test_19-32-31'
   <Type>   | <array_id>           | <array.name> | <array.shape>
   Setpoint | k2636B_smua_volt_set | volt         | (82,)
   Measured | k2636B_smua_curr     | curr         | (82,)
   Measured | k2636B_smua_volt     | volt         | (82,)
   Measured | k2636B_smub_volt     | volt         | (82,)
   Measured | k2636B_smub_curr     | curr         | (82,)
   Measured | lockin1_R            | R            | (82,)
   Measured | lockin1_P            | P            | (82,)
   Measured | lockin1_X            | X            | (82,)
Finished at 2025-11-04 19:34:45
Keyboard Interrupt


In [66]:
time_val = np.arange(0,601, 1)
print(np.shape(time_val))
print(time_val)

(601,)
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244

In [67]:
k2636B.smua.output(1)
k2636B.smub.output(1)
time.sleep(1)
data1 , plot1=npd.single_param_sweep(sr830.Time, time_val, 1, k2636B.smua.curr, sr830.R, sr830.P, sr830.X, DataName='PhotoresponseTest')
time.sleep(1)
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-11-04 19:37:33
DataSet:
   location = 'data/2025-11-04/#006_PhotoresponseTest_19-37-31'
   <Type>   | <array_id>       | <array.name> | <array.shape>
   Setpoint | lockin1_Time_set | Time         | (601,)
   Measured | k2636B_smua_curr | curr         | (601,)
   Measured | lockin1_R        | R            | (601,)
   Measured | lockin1_P        | P            | (601,)
   Measured | lockin1_X        | X            | (601,)
Finished at 2025-11-04 19:38:21
Keyboard Interrupt


In [32]:
flip_mount.move_to_state(0) #Block

volt_val = np.linspace(0.04, 1, 100)
k2636B.smub.volt(20)
k2636B.smub.output(1)
sr830.write("SLVL 1") 

sr830.write('AGAN')		  #自動檔位 sensitivity


#lOCK MEASUREMENT
k2636B.smub.volt(20)
time.sleep(1)
k2636B.smub.output(1)
time.sleep(1)
sr830.write("SLVL 0.04") 
time.sleep(5)
data1 , plot1=npd.single_param_sweep(sr830.amplitude, volt_val, 0.2, sr830.R, sr830.P, sr830.X, DataName=f'testAC_VG20_dark')
time.sleep(1)
sr830.write("SLVL 0.04")
k2636B.smub.volt(0)
k2636B.smub.output(0)
    
time.sleep(1)
flip_mount.move_to_state(0) #Block

Started at 2025-09-18 19:27:04
DataSet:
   location = 'data/2025-09-18/#085_testAC_VG20_dark_19-27-02'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (100,)
   Measured | lockin1_R             | R            | (100,)
   Measured | lockin1_P             | P            | (100,)
   Measured | lockin1_X             | X            | (100,)
Finished at 2025-09-18 19:27:36


In [56]:
sr830.write("SLVL 0.04")
k2636B.smub.volt(0)
    #k2636B.smua.output(0)
k2636B.smub.output(0)

In [53]:
k2636B.smua.volt(-3)
k2636B.smub.volt(20)

In [54]:
k2636B.smua.output(1)
k2636B.smub.output(1)

In [51]:
sr830.write('AGAN')
time.sleep(5)
sr830.write('AGAN')

In [114]:
#k2636B.smua.output(1)
k2636B.smub.volt(20)
time.sleep(1)
k2636B.smub.output(1)
time.sleep(1)
sr830.write("SLVL 2")
time.sleep(5)
sr830.write('AGAN')
time.sleep(1)
sr830.write('APHS')
time.sleep(1)
data1 , plot1=npd.single_param_sweep(sr830.Time, time_val, 1, sr830.X, sr830.R, sr830.P, DataName='PhotoresponseTest')
time.sleep(1)
sr830.write("SLVL 0.04") 
#k2636B.smua.volt(0)
k2636B.smub.volt(0)
#k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-09-18 17:46:51
DataSet:
   location = 'data/2025-09-18/#026_PhotoresponseTest_17-46-50'
   <Type>   | <array_id>       | <array.name> | <array.shape>
   Setpoint | lockin1_Time_set | Time         | (601,)
   Measured | lockin1_X        | X            | (601,)
   Measured | lockin1_R        | R            | (601,)
   Measured | lockin1_P        | P            | (601,)
Finished at 2025-09-18 17:49:03
Keyboard Interrupt


In [64]:
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

In [28]:
flip_mount.move_to_state(0) #Block


In [33]:
set_power(1.77e-3, 0.01e-3)

Measured: 0.000001 W, Error: 1.769469e-03 W
Measured: 0.000000 W, Error: 1.769629e-03 W
Measured: 0.000000 W, Error: 1.769682e-03 W
Measured: 0.000000 W, Error: 1.769842e-03 W
Measured: 0.000000 W, Error: 1.769764e-03 W
Measured: 0.000000 W, Error: 1.769789e-03 W
Measured: 0.000000 W, Error: 1.769563e-03 W
Measured: 0.000000 W, Error: 1.769817e-03 W
Measured: 0.000000 W, Error: 1.769713e-03 W
Measured: 0.000000 W, Error: 1.769717e-03 W
Measured: 0.000000 W, Error: 1.769732e-03 W
Measured: 0.000000 W, Error: 1.769695e-03 W
Measured: 0.000000 W, Error: 1.769602e-03 W
Measured: 0.000001 W, Error: 1.769391e-03 W
Measured: 0.000002 W, Error: 1.767579e-03 W
Measured: 0.000007 W, Error: 1.763262e-03 W
Measured: 0.001710 W, Error: 5.978104e-05 W
Measured: 0.001776 W, Error: -5.596870e-06 W
Target reached.
Measured: 0.001726 W, Error: 4.435880e-05 W


In [ ]:
flip_mount.move_to_state(1)
